In [ ]:
import xml.etree.cElementTree as ET
import io
import re
from transformers import AutoTokenizer
from collections import Counter
from nltk.corpus import wordnet as wn
import nltk
nltk.download('wordnet')
import pandas as pd
import numpy as np

# the semcor and omsti datasets are available at http://lcl.uniroma1.it/wsdeval/training-data

save_path = "../data/"

In [ ]:
with open(save_path + "/WSD_Evaluation_Framework/Training_Corpora/SemCor+OMSTI/semcor+omsti.gold.key.txt", "r") as f:
    gold = [line[:-1] for line in f.readlines()]
    
label_dict = {}
[label_dict.update({line.split(" ")[0]: wn.lemma_from_key(line.split(" ")[1])}) for line in gold]
    
semcor_lines = []
omsti_lines = []

with open(save_path + "WSD_Evaluation_Framework/Training_Corpora/SemCor+OMSTI/semcor+omsti.data.xml", "r") as f:
    for i, line in enumerate(f):
        if i < 877502:
            semcor_lines.append(line)
        else:
            omsti_lines.append(line)
            
print(len(semcor_lines))
print(len(omsti_lines))

## XML parsing

In [ ]:
semcor_xml = io.StringIO(("").join(semcor_lines))
omsti_xml = io.StringIO(("").join(omsti_lines))

In [ ]:
%%time

token_list = []
sentence_dict = {}

for event, entry in ET.iterparse(semcor_xml, events = ["start", "end"]):
    
    if entry.tag == "text" and event == "start":
        
        text_id = entry.attrib["id"]
        source = entry.attrib["source"]
        
    elif entry.tag == "sentence":
        
        if event == "start":
            
            sentence_id = entry.attrib["id"] 
            sentence = []
            n_tokens = 0
            
        elif event == "end":
            
            sentence_dict[sentence_id] = sentence
        
    elif entry.tag == "instance" and event == "start":
        
        if entry.text is not None:
            token = entry.text
        else:
            token = entry.attrib["lemma"]
            
        sentence.append(token)
        
        token_attrib = entry.attrib
        token_attrib["token"] = token
        token_attrib["token_num"] = n_tokens
        token_attrib["text_id"] = text_id
        token_attrib["source"] = source
        token_attrib["sentence_id"] = sentence_id
        token_list.append(token_attrib)
        
        n_tokens +=1
        
    elif entry.tag == "wf" and event == "start":
        
        if entry.text is not None:
            token = entry.text
        else:
            token = entry.attrib["lemma"]
        sentence.append(token)
        n_tokens +=1
        
        
print(len(token_list))

In [ ]:
%%time 
for event, entry in ET.iterparse(omsti_xml, events = ["start", "end"]):
    
    if entry.tag == "text" and event == "start":
        
        text_id = entry.attrib["id"]
        source = ""
        
    elif entry.tag == "sentence":
        
        if event == "start":
            
            sentence_id = entry.attrib["id"] 
            sentence = []
            n_tokens = 0
            
        elif event == "end":
            
            sentence_dict[sentence_id] = sentence
        
    elif entry.tag == "instance" and event == "start":
        
        if entry.text is not None:
            token = entry.text
        else:
            token = entry.attrib["lemma"]
        
        sentence.append(token)
        
        token_attrib = entry.attrib
        token_attrib["token"] = token
        token_attrib["token_num"] = n_tokens
        token_attrib["text_id"] = text_id
        token_attrib["source"] = source
        token_attrib["sentence_id"] = sentence_id
        token_list.append(token_attrib)
        
        n_tokens +=1
        
    elif entry.tag == "wf" and event == "start":
        
        if entry.text is not None:
            token = entry.text
        else:
            token = entry.attrib["lemma"]
        sentence.append(token)
        n_tokens +=1
        
    
print(len(token_list))

    

## Data processing and masking

In [ ]:

def fuse_sentence(sentence):
    
    sentence = " ".join(sentence).strip()
    sentence = re.sub(r"(\w'{0,2}) ([\.\,\?;:\)])", r"\1\2", sentence)
    sentnece = re.sub(r"([``\(]) (\w)", r"\1\2", sentence)
    sentence = re.sub(r"(\w) ('')", r"\1\2", sentence)
    sentence = re.sub(r"([a-zA-Z]) ('[a-zA-Z ])", r"\1\2", sentence)
    
    return sentence
    


In [ ]:

for i in range(0, len(token_list)):
    
    sublist = token_list[i]
    
    synset = label_dict[sublist["id"]].synset().name()
    
    tokens = sentence_dict[sublist["sentence_id"]]
    tokens_masked = ["[MASK]" if i == sublist["token_num"] else token for i, token in enumerate(tokens) ]
    text = fuse_sentence(tokens)
    text_masked = fuse_sentence(tokens_masked)
    
    match = re.search("\[MASK\]", text_masked)
    position = match.span()[0]
    n_characters = len(text_masked)
    relative_position = round(position/n_characters, 2)
    first_token = True if position == 0 else False
    last_token = True if position == n_characters-6 else False
    eof = True if text_masked[position:position+7] == "[MASK]." else False
    

    token_list[i]["text"] = text
    token_list[i]["text_masked"] = text_masked
    token_list[i]["position"] = position
    token_list[i]["relative_position"] = relative_position
    token_list[i]["n_tokens"] = n_tokens
    token_list[i]["n_characters"] = n_characters
    token_list[i]["first-token"] = first_token
    token_list[i]["last-token"] =  last_token
    token_list[i]["end-of-sentence"] = eof
    token_list[i]["synset"] = synset
    
df = pd.DataFrame(token_list)

In [ ]:
# filtering out longer sequences
df = df.loc[df["n_characters"]<= 500].reset_index(drop = True)

## Selecting and saving datasets

In [ ]:
# selection of nouns
filter_name = "noun-synsets-strat"
synset_filter= ["person.n.01", "manner.n.01", "line.n.16"]
df_nouns = df.loc[df["synset"].isin(synset_filter)]
sample_size = df_nouns.groupby(["synset"]).size().min()
df_nouns = df_nouns.groupby(["synset"]).sample(sample_size, random_state = 42).sort_index()
df["noun-synsets-strat"] = False
df.loc[df_nouns.index, "noun-synsets-strat"] = True


# filter for the synset of person.n.01
filter_name = "person.n.01"
df["person.n.01"] = False
df.loc[df["synset"] == "person.n.01", "person.n.01"] = True


In [ ]:
# only keeping instances that are in specified datasets
df = df.loc[np.where(df[["noun-synsets", "noun-synsets-strat", "person.n.01"]])[0]].reset_index(drop= True)
df.drop_duplicates(inplace = True)

In [ ]:
## Saving

df.to_csv(f"{lrz_path}datasets/semcor&omsti_df.csv", encoding = "utf8")

with open(lrz_path + "datasets/semcor&omsti_texts.txt", "w",encoding = "utf8")as f:
        
        [f.write(text + "\n") for text in df["text_masked"]]

